# 第16章：综合实战 — 从 GPT-2 到现代架构

## 本章目标
- 将 Ch11-15 的所有改进整合到一个模型中
- 在相同数据上对比 GPT-2 架构 vs 现代架构的训练效果
- 回顾 16 章学习路线，规划后续方向

## 前置知识
- 完成第11-15章的学习
- 理解 RMSNorm、SwiGLU、RoPE、GQA、MoE、GRPO 的原理

## 整合清单

| 改进 | 来源 | 替换 |
|------|------|------|
| RMSNorm | Ch11 | LayerNorm |
| SwiGLU | Ch11 | GELU FFN |
| RoPE | Ch11 | 绝对位置编码 |
| GQA | Ch11 | MHA |
| MoE | Ch12 | Dense FFN |

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib
    !wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## 1. 架构升级：复制 Ch11 的核心组件

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

# === Ch11 组件 ===

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class SwiGLUFFN(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        hidden = int(2/3 * 4 * n_embd)
        hidden = ((hidden + 255) // 256) * 256
        self.w_gate = nn.Linear(n_embd, hidden, bias=False)
        self.w_up   = nn.Linear(n_embd, hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, freqs)
    freqs = torch.cat([freqs, freqs], dim=-1)
    return freqs.cos(), freqs.sin()

def apply_rotary_emb(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    return x * cos + torch.cat([-x2, x1], dim=-1) * sin

print("Ch11 核心组件已加载：RMSNorm, SwiGLUFFN, RoPE")

In [ ]:
class GQAWithRoPE(nn.Module):
    """GQA + RoPE 整合版。"""
    def __init__(self, n_embd, n_head, n_kv_heads, block_size, dropout=0.1):
        super().__init__()
        self.n_head = n_head
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_head // n_kv_heads
        self.head_dim = n_embd // n_head
        self.wq = nn.Linear(n_embd, n_embd, bias=False)
        self.wk = nn.Linear(n_embd, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(n_embd, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x, rope_cos=None, rope_sin=None):
        B, T, C = x.size()
        q = self.wq(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        if rope_cos is not None:
            q = apply_rotary_emb(q, rope_cos, rope_sin)
            k = apply_rotary_emb(k, rope_cos, rope_sin)
        if self.n_rep > 1:
            k = k.unsqueeze(2).expand(B, self.n_kv_heads, self.n_rep, T, self.head_dim).reshape(B, self.n_head, T, self.head_dim)
            v = v.unsqueeze(2).expand(B, self.n_kv_heads, self.n_rep, T, self.head_dim).reshape(B, self.n_head, T, self.head_dim)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(self.attn_dropout(att), dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.wo(y))

print("GQA + RoPE 模块已定义")

In [ ]:
class ModernBlock(nn.Module):
    """RMSNorm + GQA(+RoPE) + SwiGLU Block"""
    def __init__(self, n_embd, n_head, n_kv_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GQAWithRoPE(n_embd, n_head, n_kv_heads, block_size, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.ffn = SwiGLUFFN(n_embd, dropout=dropout)
    def forward(self, x, rope_cos=None, rope_sin=None):
        x = x + self.attn(self.ln_1(x), rope_cos, rope_sin)
        x = x + self.ffn(self.ln_2(x))
        return x

class ModernGPT(nn.Module):
    """ModernGPT: RMSNorm + SwiGLU + RoPE + GQA"""
    def __init__(self, vocab_size, n_embd=64, n_head=4, n_kv_heads=2,
                 n_layer=4, block_size=64, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([
            ModernBlock(n_embd, n_head, n_kv_heads, block_size, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight
        head_dim = n_embd // n_head
        cos_c, sin_c = precompute_rope_freqs(head_dim, block_size)
        self.register_buffer("rope_cos", cos_c)
        self.register_buffer("rope_sin", sin_c)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        x = self.drop(self.wte(idx))
        cos = self.rope_cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.rope_sin[:T].unsqueeze(0).unsqueeze(0)
        for block in self.h:
            x = block(x, cos, sin)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# 验证
model = ModernGPT(vocab_size=65, n_embd=64, n_head=4, n_kv_heads=2, n_layer=4, block_size=64)
params = sum(p.numel() for p in model.parameters())
print(f"ModernGPT 参数量: {params:,} ({params/1e6:.2f}M)")
x = torch.randint(0, 65, (2, 32))
logits, loss = model(x, targets=x)
print(f"logits: {logits.shape}, loss: {loss.item():.4f}")

## 2. 添加 MoE（来自 Ch12）

In [ ]:
class MoELayer(nn.Module):
    """Top-k MoE FFN（来自 Ch12）。"""
    def __init__(self, n_embd, num_experts=4, top_k=2, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            SwiGLUFFN(n_embd, dropout=dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)  # (B*T, C)
        router_logits = self.router(x_flat)  # (B*T, num_experts)
        weights, indices = torch.topk(router_logits, self.top_k, dim=-1)
        weights = F.softmax(weights, dim=-1)
        
        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            expert_idx = indices[:, k]  # (B*T,)
            w = weights[:, k:k+1]  # (B*T, 1)
            for e in range(self.num_experts):
                mask = (expert_idx == e)
                if mask.any():
                    expert_out = self.experts[e](x_flat[mask])
                    output[mask] += w[mask] * expert_out
        return output.view(B, T, C)

class ModernMoEBlock(nn.Module):
    """ModernBlock + MoE 替换 FFN"""
    def __init__(self, n_embd, n_head, n_kv_heads, block_size, num_experts=4, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GQAWithRoPE(n_embd, n_head, n_kv_heads, block_size, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.moe = MoELayer(n_embd, num_experts=num_experts, top_k=2, dropout=dropout)

    def forward(self, x, rope_cos=None, rope_sin=None):
        x = x + self.attn(self.ln_1(x), rope_cos, rope_sin)
        x = x + self.moe(self.ln_2(x))
        return x

print("MoE 模块已定义")

In [ ]:
# 参数量对比：Dense vs MoE
dense_params = sum(p.numel() for p in ModernGPT(65, n_embd=64, n_head=4, n_kv_heads=2, n_layer=4, block_size=64).parameters())

# MoE 版本：用 MoE block 替换部分 FFN
moe_model = ModernGPT(65, n_embd=64, n_head=4, n_kv_heads=2, n_layer=4, block_size=64)
# 替换后 2 层为 MoE
for i in range(2, 4):
    moe_model.h[i] = ModernMoEBlock(64, 4, 2, 64, num_experts=4)
moe_params = sum(p.numel() for p in moe_model.parameters())

print(f"{'模型':<25s} {'总参数量':>12s} {'活跃参数量':>12s}")
print(f"{'ModernGPT (Dense)':<25s} {dense_params:>12,} {dense_params:>12,}")
print(f"{'ModernGPT + MoE':<25s} {moe_params:>12,} {'~' + str(dense_params):>12s}")
print(f"\nMoE 增加 {moe_params - dense_params:,} 参数（更多专家容量），")
print(f"但每次前向传播只激活 top-2 专家，计算量接近 Dense 版本")

## 3. 训练对比：GPT-2 vs ModernGPT

在相同数据（Shakespeare）和相同计算预算下对比两个架构。

In [ ]:
# GPT-2 基线模型（来自 Ch1）
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class GPT2MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class GPT2Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = GPT2MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT2(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=4, block_size=64, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GPT2Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight
        self.apply(lambda m: nn.init.normal_(m.weight, mean=0.0, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

print("GPT-2 基线模型已定义")

In [ ]:
# 加载 Shakespeare 数据
import os
if not os.path.exists('input.txt'):
    !wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
data = torch.tensor(encode(text), dtype=torch.long)

print(f"数据量: {len(data):,} tokens")
print(f"词表大小: {vocab_size} (字符级)")
print(f"前100字符: {text[:100]}...")

In [ ]:
# 训练函数
def train_model(model, data, vocab_size, steps=500, block_size=64, batch_size=32, lr=3e-4, eval_every=50):
    """训练循环，返回 loss 历史。"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    
    for step in range(steps):
        # 随机采样 batch
        ix = torch.randint(len(data) - block_size, (batch_size,))
        x = torch.stack([data[i:i+block_size] for i in ix])
        y = torch.stack([data[i+1:i+block_size+1] for i in ix])
        
        _, loss = model(x, targets=y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (step + 1) % eval_every == 0:
            losses.append((step + 1, loss.item()))
            print(f"  Step {step+1:4d}: loss={loss.item():.4f}")
    
    return losses

print("训练函数已定义")

In [ ]:
# 训练 GPT-2 基线
print("=== 训练 GPT-2 基线 ===")
torch.manual_seed(42)
gpt2_model = GPT2(vocab_size, n_embd=64, n_head=4, n_layer=4, block_size=64)
print(f"参数量: {sum(p.numel() for p in gpt2_model.parameters()):,}")
gpt2_losses = train_model(gpt2_model, data, vocab_size, steps=500, block_size=64, batch_size=32)

In [ ]:
# 训练 ModernGPT
print("=== 训练 ModernGPT ===")
torch.manual_seed(42)
modern_model = ModernGPT(vocab_size, n_embd=64, n_head=4, n_kv_heads=2, n_layer=4, block_size=64)
print(f"参数量: {sum(p.numel() for p in modern_model.parameters()):,}")
modern_losses = train_model(modern_model, data, vocab_size, steps=500, block_size=64, batch_size=32)

In [ ]:
# 对比 loss 曲线
gpt2_steps, gpt2_vals = zip(*gpt2_losses)
modern_steps, modern_vals = zip(*modern_losses)

plt.figure(figsize=(10, 5))
plt.plot(gpt2_steps, gpt2_vals, 'b-o', label='GPT-2 (LayerNorm + GELU + AbsPE + MHA)')
plt.plot(modern_steps, modern_vals, 'r-s', label='ModernGPT (RMSNorm + SwiGLU + RoPE + GQA)')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('训练对比：GPT-2 vs ModernGPT (Shakespeare, 500 steps)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n最终 Loss 对比:")
print(f"  GPT-2:     {gpt2_vals[-1]:.4f}")
print(f"  ModernGPT: {modern_vals[-1]:.4f}")
diff = gpt2_vals[-1] - modern_vals[-1]
print(f"  差异: {diff:+.4f} ({'ModernGPT 更好' if diff > 0 else 'GPT-2 更好'})")
print("\n注意：baby 模型下差异可能不大，但在大规模模型和更多训练数据上，")
print("现代架构的优势会更加明显（更好的扩展性和参数效率）。")

## 4. GRPO 对齐（概念演示）

将 Ch14 的 GRPO 应用到 ModernGPT 上的概念代码。实际运行需要 Colab T4 GPU。

```python
# 概念代码（需要 GPU 环境）
from trl import GRPOTrainer
from peft import LoraConfig

# 1. 加载预训练模型
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")

# 2. 配置 LoRA
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"])

# 3. 定义奖励函数（数学题正确性）
def reward_fn(completions, **kwargs):
    rewards = []
    for comp, answer in zip(completions, kwargs["answer"]):
        try:
            pred = comp.split("####")[-1].strip()
            rewards.append(1.0 if pred == answer else 0.0)
        except:
            rewards.append(0.0)
    return rewards

# 4. 配置并启动 GRPO 训练
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=gsm8k_dataset,
    reward_funcs=[reward_fn],
    peft_config=lora_config,
)
trainer.train()
```

参考：[DeepSeek-R1](https://arxiv.org/abs/2501.12948), [trl GRPOTrainer](https://huggingface.co/docs/trl/main_classes/grpo_trainer)

## 5. 路线图：16 章学习回顾与未来方向

### 学习路线回顾

| Part | 章节 | 核心内容 |
|------|------|----------|
| **Part I: 基础架构与预训练** | Ch1-4 | GPT架构、Tokenizer、预训练、分布式训练 |
| **Part II: 对齐训练** | Ch5-8 | SFT、Reward Model、RLHF/PPO、DPO |
| **Part III: 生产部署** | Ch9-10 | 推理优化、完整Pipeline |
| **Part IV: 走向SOTA** | Ch11-16 | 现代架构、MoE、预训练工程、GRPO、长上下文 |

### 后续学习方向

1. **模型蒸馏 (Distillation)**
   - 用大模型的知识训练小模型
   - 参考：DeepSeek-R1 的蒸馏实验

2. **多模态 (Multimodal)**
   - 视觉-语言模型 (VLM)：LLaVA, Qwen-VL
   - 音频、视频理解

3. **Agent 框架**
   - 工具使用 (Tool Use)、规划 (Planning)
   - RAG、代码执行、搜索

4. **端侧部署**
   - 量化 (GPTQ, AWQ, GGUF)
   - 剪枝 (Pruning)
   - 移动端/嵌入式部署

5. **持续预训练与领域适配**
   - 领域特定数据混合
   - LoRA / QLoRA 微调实践

## 练习

1. 增加训练步数到 2000，观察 GPT-2 和 ModernGPT 的 loss 差异是否更加明显
2. 尝试将 MoE 添加到 ModernGPT 的所有层，对比训练效果
3. 在 Colab T4 上用真实 Qwen2.5-0.5B 运行 GRPO 训练

## 延伸阅读

- [nanoGPT](https://github.com/karpathy/nanoGPT) — 本教程的参考实现
- [DeepSeek-V3](https://arxiv.org/abs/2412.19437) — 现代 MoE 架构
- [DeepSeek-R1](https://arxiv.org/abs/2501.12948) — GRPO 推理训练
- [Llama 3 技术报告](https://ai.meta.com/blog/meta-llama-3/) — 工业级训练细节
- [HuggingFace transformers](https://github.com/huggingface/transformers) — 模型实现参考